Drive + Configurações Gerais

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

BASE_IN = "/content/drive/MyDrive/Mestrado/Dados Gerais"
BASE_OUT = "/content/drive/MyDrive/Mestrado/Dados_Anonimizados"

ANOS = [2018, 2019, 2020, 2021, 2022, 2023]

LAT_COL = "latitude"
LON_COL = "longitude"

os.makedirs(BASE_OUT, exist_ok=True)

print("📂 Origem :", BASE_IN)
print("📂 Destino:", BASE_OUT)
print("📅 Anos   :", ANOS)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Origem : /content/drive/MyDrive/Mestrado/Dados Gerais
📂 Destino: /content/drive/MyDrive/Mestrado/Dados_Anonimizados
📅 Anos   : [2018, 2019, 2020, 2021, 2022, 2023]


scripts de anonimização

In [11]:
import numpy as np
import pandas as pd

def _planar_laplace_noise_vector(n, epsilon, rng):
    """
    Gera n deslocamentos (dx, dy) em METROS conforme o mecanismo
    Planar Laplace (geo-indistinguishability) de Andrés et al. (2013).

    Garantia formal: para dois pontos x, x' separados por distância d (em km),
    o output do mecanismo satisfaz (ε·d)-privacidade diferencial.

    Interpretação prática do parâmetro epsilon (em unidades de privacidade/km):
        ε = 0.1  → mediana ~20 km  (privacidade muito alta)
        ε = 1.0  → mediana ~2 km   (privacidade moderada)
        ε = 5.0  → mediana ~400 m  (privacidade baixa)

    Parâmetros:
        n       : número de pontos
        epsilon : parâmetro de privacidade (positivo, em 1/km)
        rng     : np.random.Generator (obrigatório, para reprodutibilidade controlada)

    Retorna:
        (dx, dy) : arrays de deslocamentos em metros
    """
    if epsilon <= 0:
        raise ValueError(f"epsilon deve ser positivo, recebeu {epsilon}")

    # Ângulo uniforme em [0, 2π)
    theta = rng.uniform(0, 2*np.pi, size=n)

    # Raio segue distribuição Gamma(k=2, scale=1/ε) em km.
    # Isso é equivalente à inversa de W-Lambert da fórmula original
    # de Andrés et al. (2013) quando se gera ruído polar.
    r_km = rng.gamma(shape=2.0, scale=1.0/float(epsilon), size=n)
    r_m = r_km * 1000.0  # converte para metros

    dx = r_m * np.cos(theta)
    dy = r_m * np.sin(theta)
    return dx, dy


def add_planar_laplace_noise(df, epsilon, seed,
                             lat_col="latitude", lon_col="longitude"):
    """
    Aplica ruído Planar Laplace (geo-indistinguishability) às coordenadas.

    Parâmetros:
        df      : DataFrame com colunas lat_col e lon_col
        epsilon : parâmetro de privacidade (1/km). Sem default — explícito sempre.
        seed    : semente do RNG. Sem default — explícito sempre.
        lat_col : nome da coluna de latitude (default "latitude")
        lon_col : nome da coluna de longitude (default "longitude")

    Retorna:
        DataFrame com as mesmas linhas, mas com lat/lon perturbadas.
    """
    rng = np.random.default_rng(seed)
    out = df.copy()

    # Sanitiza colunas
    lat = pd.to_numeric(out[lat_col], errors="coerce")
    lon = pd.to_numeric(out[lon_col], errors="coerce")
    mask = lat.notna() & lon.notna()
    out = out.loc[mask].copy()

    lat_vals = out[lat_col].astype(float).values
    lon_vals = out[lon_col].astype(float).values

    n = len(out)
    dx_m, dy_m = _planar_laplace_noise_vector(n=n, epsilon=epsilon, rng=rng)

    # Conversão metros → graus
    # 1° lat ≈ 111_320 m em qualquer lugar
    # 1° lon ≈ 111_320 * cos(lat) m
    meters_per_deg_lat = 111_320.0
    lat_rad = np.deg2rad(lat_vals)
    meters_per_deg_lon = 111_320.0 * np.cos(lat_rad)
    meters_per_deg_lon = np.where(meters_per_deg_lon == 0, 1e-9, meters_per_deg_lon)

    dlat = dy_m / meters_per_deg_lat
    dlon = dx_m / meters_per_deg_lon

    out[lat_col] = lat_vals + dlat
    out[lon_col] = lon_vals + dlon

    return out

In [12]:
# === Validação rápida do novo mecanismo Planar Laplace ===
# Gera 10_000 deslocamentos com diferentes epsilons e mostra
# a distribuição radial. Espera-se:
#   - mediana de epsilon=1.0 em torno de 1.7 km
#   - mediana de epsilon=0.1 em torno de 17 km
#   - mediana de epsilon=5.0 em torno de 0.34 km
# Isso confirma que ε agora está em "1/km" como na literatura.

rng_test = np.random.default_rng(123)

for eps in [0.1, 0.5, 1.0, 2.0, 5.0]:
    dx, dy = _planar_laplace_noise_vector(n=10_000, epsilon=eps, rng=rng_test)
    r = np.sqrt(dx**2 + dy**2)
    print(f"ε={eps:>4}  →  mediana = {np.median(r):>8.1f} m   "
          f"P95 = {np.percentile(r, 95):>8.1f} m   "
          f"max = {r.max():>9.1f} m")

ε= 0.1  →  mediana =  16867.5 m   P95 =  48001.3 m   max =  115776.2 m
ε= 0.5  →  mediana =   3327.4 m   P95 =   9288.2 m   max =   26418.8 m
ε= 1.0  →  mediana =   1682.6 m   P95 =   4721.1 m   max =   12392.4 m
ε= 2.0  →  mediana =    842.5 m   P95 =   2369.6 m   max =    5907.7 m
ε= 5.0  →  mediana =    340.6 m   P95 =    962.0 m   max =    2327.0 m


In [13]:
def permutacao(df, seed):
    """
    Permuta latitude e longitude de forma independente entre todos os registros.
    Embaralha lat e lon SEPARADAMENTE — o que significa que cada registro
    recebe lat de algum outro registro aleatório e lon de outro registro aleatório
    (não necessariamente o mesmo).

    Estocástica: aceita seed para reprodutibilidade.
    """
    rng = np.random.default_rng(seed)
    out = df.copy()
    out[LAT_COL] = rng.permutation(out[LAT_COL].values)
    out[LON_COL] = rng.permutation(out[LON_COL].values)
    return out


def generalizacao(df, casas):
    """
    Generalização por arredondamento decimal.

    Interpretação aproximada das casas decimais em latitude/longitude
    no extremo sul do Brasil (latitude ~ -30°):
        casas=1  → célula ~11 km
        casas=2  → célula ~1.1 km
        casas=3  → célula ~110 m
        casas=4  → célula ~11 m

    Determinística: NÃO usa seed.
    """
    out = df.copy()
    out[LAT_COL] = out[LAT_COL].round(casas)
    out[LON_COL] = out[LON_COL].round(casas)
    return out


def microagregacao(df, k, seed):
    """
    Microagregação por agrupamento aleatório de k registros.
    Cada grupo recebe a coordenada média do próprio grupo.

    Implementação clássica para atingir k-anonimato em dados numéricos
    (Domingo-Ferrer & Mateo-Sanz, 2002).

    Estocástica: o seed controla o embaralhamento inicial, que determina
    quais registros caem juntos em cada grupo. Mesmo seed = mesmos grupos.
    """
    rng = np.random.default_rng(seed)
    # Embaralha índices reproduzivelmente
    idx = rng.permutation(len(df))
    out = df.iloc[idx].reset_index(drop=True).copy()

    for i in range(0, len(out), k):
        grupo = out.iloc[i:i+k]
        out.loc[i:i+k-1, LAT_COL] = grupo[LAT_COL].mean()
        out.loc[i:i+k-1, LON_COL] = grupo[LON_COL].mean()
    return out


def privacidade_diferencial(df, epsilon, seed):
    """
    Aplica geo-indistinguishability (Andrés et al. 2013) às coordenadas.

    Parâmetros:
        epsilon : parâmetro de privacidade em 1/km (positivo).
                  Menor = mais privacidade, mais ruído.
        seed    : semente do RNG, obrigatória para reprodutibilidade.
    """
    out = df.copy()
    out[LAT_COL] = pd.to_numeric(out[LAT_COL], errors="coerce")
    out[LON_COL] = pd.to_numeric(out[LON_COL], errors="coerce")
    out = out.dropna(subset=[LAT_COL, LON_COL])

    out = add_planar_laplace_noise(
        out,
        epsilon=epsilon,
        seed=seed,
        lat_col=LAT_COL,
        lon_col=LON_COL
    )
    return out



In [14]:
# ============================================================
# Configuração das técnicas e variantes (parâmetros)
# ============================================================

# Generalização: arredondamento decimal.
# Determinística — uma única variante por número de casas.
GENERALIZACAO_CASAS = [1, 2, 3, 4]

# Microagregação: agrupamento aleatório.
# Estocástica — k controla o tamanho dos grupos.
MICROAGREGACAO_KS = [2, 5, 10]

# Privacidade Diferencial: parâmetro ε em 1/km (Andrés et al. 2013).
# Estocástica — mais ε = menos privacidade, menos ruído.
EPSILONS_DP = [0.1, 0.5, 1.0, 2.0, 5.0]

# Permutação: estocástica, não tem parâmetro além do seed.
# (Apenas uma variante.)

# ------------------------------------------------------------
# Funções estocásticas RECEBEM seed; determinísticas NÃO.
# Cada entrada do dicionário abaixo é (nome, função_de_aplicar)
# que recebe (df, seed) — para uniformidade, mesmo as
# determinísticas ignoram o seed internamente.
# ------------------------------------------------------------

TECNICAS = {}

# Permutação (estocástica, 1 variante)
TECNICAS["permutacao"] = lambda df, seed: permutacao(df, seed=seed)

# Generalização (determinística, 4 variantes)
for casas in GENERALIZACAO_CASAS:
    nome = f"generalizacao_dec{casas}"
    TECNICAS[nome] = (lambda df, seed, c=casas: generalizacao(df, casas=c))

# Microagregação (estocástica, 3 variantes)
for k in MICROAGREGACAO_KS:
    nome = f"microagregacao_k{k}"
    TECNICAS[nome] = (lambda df, seed, k_=k: microagregacao(df, k=k_, seed=seed))

# Privacidade Diferencial (estocástica, 5 variantes)
for eps in EPSILONS_DP:
    nome = f"dp_eps_{eps}"
    TECNICAS[nome] = (lambda df, seed, e=eps: privacidade_diferencial(df, epsilon=e, seed=seed))

# Marcação de quais técnicas são estocásticas
# (importante para a Fase 2: só elas variam entre seeds)
TECNICAS_ESTOCASTICAS = (
    {"permutacao"}
    | {f"microagregacao_k{k}" for k in MICROAGREGACAO_KS}
    | {f"dp_eps_{eps}" for eps in EPSILONS_DP}
)

print(f"📊 Total de técnicas configuradas: {len(TECNICAS)}")
print(f"   - Permutação:           1 variante")
print(f"   - Generalização:        {len(GENERALIZACAO_CASAS)} variantes")
print(f"   - Microagregação:       {len(MICROAGREGACAO_KS)} variantes")
print(f"   - Privacidade Dif.:     {len(EPSILONS_DP)} variantes")
print(f"   Estocásticas: {len(TECNICAS_ESTOCASTICAS)} / {len(TECNICAS)}")
print()
print("Nomes:")
for nome in TECNICAS:
    marker = "🎲" if nome in TECNICAS_ESTOCASTICAS else "🔒"
    print(f"   {marker} {nome}")

📊 Total de técnicas configuradas: 13
   - Permutação:           1 variante
   - Generalização:        4 variantes
   - Microagregação:       3 variantes
   - Privacidade Dif.:     5 variantes
   Estocásticas: 9 / 13

Nomes:
   🎲 permutacao
   🔒 generalizacao_dec1
   🔒 generalizacao_dec2
   🔒 generalizacao_dec3
   🔒 generalizacao_dec4
   🎲 microagregacao_k2
   🎲 microagregacao_k5
   🎲 microagregacao_k10
   🎲 dp_eps_0.1
   🎲 dp_eps_0.5
   🎲 dp_eps_1.0
   🎲 dp_eps_2.0
   🎲 dp_eps_5.0


In [ ]:
# ============================================================
# Loop principal: gera versões anonimizadas em Parquet empilhado
# Estrutura: BASE_OUT/{tecnica}/{ano}.parquet
# Cada Parquet contém todas as N_SEEDS empilhadas (coluna 'seed')
# ============================================================

# Garante pyarrow (Colab geralmente já tem)
try:
    import pyarrow  # noqa
except ImportError:
    !pip install -q pyarrow

from tqdm.auto import tqdm

# Quantos restarts por técnica estocástica
N_SEEDS = 20

# Se True, pula combinações ano+técnica que já têm .parquet salvo
RESUMIR = True

# Tamanho típico de DataFrame por seed (linhas)
# Útil para estimar memória durante o concat
def _aplicar_tecnica_todas_seeds(df_orig, nome, func, n_seeds):
    """
    Aplica `func` ao DataFrame original N vezes (uma por seed) e retorna
    um DataFrame empilhado com coluna 'seed' identificando cada run.
    Para técnicas determinísticas (não em TECNICAS_ESTOCASTICAS), aplica
    apenas uma vez com seed=0.
    """
    seeds = list(range(n_seeds)) if nome in TECNICAS_ESTOCASTICAS else [0]
    pedacos = []
    for s in seeds:
        df_anon = func(df_orig, seed=s).copy()
        df_anon["seed"] = s
        pedacos.append(df_anon)
    return pd.concat(pedacos, ignore_index=True)


# Cria diretórios de saída
os.makedirs(f"{BASE_OUT}/original", exist_ok=True)
for nome in TECNICAS:
    os.makedirs(f"{BASE_OUT}/{nome}", exist_ok=True)


print(f"🔧 Configuração:")
print(f"   N_SEEDS = {N_SEEDS}")
print(f"   Técnicas: {len(TECNICAS)} ({len(TECNICAS_ESTOCASTICAS)} estocásticas)")
print(f"   Anos: {ANOS}")
print(f"   Retomar de checkpoints: {RESUMIR}")
print()

# Loop principal
for ano in ANOS:
    print(f"\n{'='*60}")
    print(f"📅 ANO {ano}")
    print('='*60)

    in_path = f"{BASE_IN}/{ano}_COMPLETO.csv"
    print(f"📥 Lendo {in_path}")
    df = pd.read_csv(in_path, sep=";", low_memory=False)
    df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")
    df[LON_COL] = pd.to_numeric(df[LON_COL], errors="coerce")
    df = df.dropna(subset=[LAT_COL, LON_COL]).reset_index(drop=True)
    print(f"   {len(df):,} registros válidos")

    # Salva original (1× só, com seed=0 por uniformidade)
    out_path = f"{BASE_OUT}/original/{ano}.parquet"
    if RESUMIR and os.path.exists(out_path):
        print(f"   ⏭️  original já existe")
    else:
        df_orig = df.copy()
        df_orig["seed"] = 0
        df_orig.to_parquet(out_path, compression="snappy", index=False)
        print(f"   ✅ original salvo")

    # Aplica cada técnica
    for nome, func in tqdm(TECNICAS.items(), desc=f"  Técnicas {ano}", leave=False):
        out_path = f"{BASE_OUT}/{nome}/{ano}.parquet"

        if RESUMIR and os.path.exists(out_path):
            continue

        df_anon_stacked = _aplicar_tecnica_todas_seeds(df, nome, func, N_SEEDS)
        df_anon_stacked.to_parquet(out_path, compression="snappy", index=False)
        # marca como concluído em mensagem curta
        n_seeds_usadas = 1 if nome not in TECNICAS_ESTOCASTICAS else N_SEEDS
        tqdm.write(f"   ✅ {nome}: {n_seeds_usadas} seeds × {len(df)} pontos")

print(f"\n{'='*60}")
print("🎉 Geração de dados anonimizados concluída!")
print('='*60)

# Verificação final: lista o que foi gerado
print("\n📦 Inventário:")
for nome in ["original"] + list(TECNICAS.keys()):
    pasta = f"{BASE_OUT}/{nome}"
    arquivos = sorted([f for f in os.listdir(pasta) if f.endswith(".parquet")])
    tamanho_mb = sum(os.path.getsize(f"{pasta}/{f}") for f in arquivos) / 1024**2
    print(f"   {nome:<30s} → {len(arquivos)} anos, {tamanho_mb:>6.1f} MB")

🔧 Configuração:
   N_SEEDS = 20
   Técnicas: 13 (9 estocásticas)
   Anos: [2018, 2019, 2020, 2021, 2022, 2023]
   Retomar de checkpoints: True


📅 ANO 2018
📥 Lendo /content/drive/MyDrive/Mestrado/Dados Gerais/2018_COMPLETO.csv
   56,529 registros válidos
   ✅ original salvo


  Técnicas 2018:   0%|          | 0/13 [00:00<?, ?it/s]

   ✅ permutacao: 20 seeds × 56529 pontos
   ✅ generalizacao_dec1: 1 seeds × 56529 pontos
   ✅ generalizacao_dec2: 1 seeds × 56529 pontos
   ✅ generalizacao_dec3: 1 seeds × 56529 pontos
   ✅ generalizacao_dec4: 1 seeds × 56529 pontos
   ✅ microagregacao_k2: 20 seeds × 56529 pontos
   ✅ microagregacao_k5: 20 seeds × 56529 pontos
   ✅ microagregacao_k10: 20 seeds × 56529 pontos
   ✅ dp_eps_0.1: 20 seeds × 56529 pontos
   ✅ dp_eps_0.5: 20 seeds × 56529 pontos
   ✅ dp_eps_1.0: 20 seeds × 56529 pontos
   ✅ dp_eps_2.0: 20 seeds × 56529 pontos
   ✅ dp_eps_5.0: 20 seeds × 56529 pontos

📅 ANO 2019
📥 Lendo /content/drive/MyDrive/Mestrado/Dados Gerais/2019_COMPLETO.csv
   71,939 registros válidos
   ✅ original salvo


  Técnicas 2019:   0%|          | 0/13 [00:00<?, ?it/s]

   ✅ permutacao: 20 seeds × 71939 pontos
   ✅ generalizacao_dec1: 1 seeds × 71939 pontos
   ✅ generalizacao_dec2: 1 seeds × 71939 pontos
   ✅ generalizacao_dec3: 1 seeds × 71939 pontos
   ✅ generalizacao_dec4: 1 seeds × 71939 pontos
